# 🌫️ EDA — Karachi AQI Predictor (Pollutant-First)

**Approach:** This project follows a **pollutant-first AQI forecasting** approach.  
Instead of predicting OpenWeather's 1–5 AQI category directly, the system:

1. Forecasts **pollutant concentrations** (PM2.5, PM10, O3, NO2) at +24h / +48h / +72h
2. Converts each to an **AQI sub-index** using EPA breakpoint interpolation  
3. The **final AQI = max sub-index** (dominant pollutant convention)

> ⚠️ This EDA uses hourly pollutant values as approximations for official averaging windows  
> (24h PM, 8h O3). Educational use only — not regulatory-grade.

---
**All data loads from MongoDB Atlas (raw_data + features collections).**

**Goal:** Explore 90 days of hourly OpenWeather air quality + meteorological data for Karachi,  
understand patterns, validate engineered features, and justify modelling decisions.

| | |
|---|---|
| **City** | Karachi, Pakistan (24.86°N, 67.00°E) |
| **Source** | OpenWeather Air Pollution API + Open-Meteo weather |
| **Feature Store** | **MongoDB Atlas** — all data is loaded directly from the cloud database |
| **Resolution** | Hourly |
| **Backfill** | ~90 days |
| **AQI Scale** | 1 Good → 5 Very Poor (OpenWeather) |

---

## ⚙️ Prerequisites — Before Running This Notebook

All data lives in **MongoDB Atlas** (no local CSV files).  
You need valid credentials in a `.env` file at the project root:

```
OPENWEATHER_API_KEY=your_key
MONGODB_URI=mongodb+srv://<user>:<password>@<cluster>.mongodb.net/...
DB_NAME=aqi_predictor
```

**Steps:**
1. Copy `.env.example` → `.env` and fill in your values.
2. Make sure `backfill.py` has been run at least once so the database is populated.
3. Run all cells top to bottom.

> If you cannot connect to MongoDB, skip to the **"CSV Fallback"** cell which lets you
> export/import data from a local file instead.

---

### Table of Contents
1. [Setup & Data Loading](#1)
2. [Dataset Overview](#2)
3. [Missing Value Analysis](#3)
4. [AQI Distribution & Category Breakdown](#4)
5. [AQI Trend over Time (with rolling average)](#5)
6. [Daily Average AQI — Colour-coded Bar Chart](#6)
7. [All Pollutants over Time](#7)
8. [PM2.5 & PM10 with WHO Guidelines](#8)
9. [Meteorological Features over Time](#9)
10. [AQI by Hour of Day — Box Plot](#10)
11. [Hour × Weekday Heatmap](#11)
12. [AQI by Weekday — Violin Plot](#12)
13. [Monthly/Daily AQI Heatmap (Calendar View)](#13)
14. [Correlation Heatmap](#14)
15. [Pollutants vs AQI — Scatter + Regression](#15)
16. [Wind Speed & Direction vs AQI](#16)
17. [Temperature & Humidity vs AQI](#17)
18. [Pair Plot — Top Correlated Features](#18)
19. [AQI Autocorrelation (Lag Analysis)](#19)
20. [Engineered Lag & Rolling Features](#20)
21. [AQI Change Rate Distribution](#21)
22. [Target Variable Distributions](#22)
23. [Quick Feature Importance Preview](#23)
24. [Key Insights & Modelling Implications](#24)


## 1. Setup & Data Loading <a id='1'></a>

In [ ]:
import sys, os
from pathlib import Path

# ── Point Python to the project root so `src.*` imports resolve ────────────────
PROJECT_ROOT = Path(os.path.abspath(".."))
sys.path.insert(0, str(PROJECT_ROOT))

# ── Load .env FIRST — must happen before any src.config import ────────────────
# python-dotenv reads MONGODB_URI, OPENWEATHER_API_KEY, DB_NAME etc.
from dotenv import load_dotenv
env_path = PROJECT_ROOT / ".env"
if env_path.exists():
    load_dotenv(dotenv_path=env_path)
    print(f"✅ .env loaded from {env_path}")
else:
    print(f"⚠️  No .env file found at {env_path}")
    print("   Set MONGODB_URI and DB_NAME as environment variables, or create .env")

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Style ──────────────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
PLOTLY_TEMPLATE = "plotly_white"
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (14, 5),
                     "axes.spines.top": False, "axes.spines.right": False})

# ── AQI palette ────────────────────────────────────────────────────────────────
AQI_COLORS = {1: "#00e400", 2: "#ffff00", 3: "#ff7e00", 4: "#ff0000", 5: "#8f3f97"}
AQI_LABELS = {1: "Good", 2: "Fair", 3: "Moderate", 4: "Poor", 5: "Very Poor"}

print("✅ Libraries loaded.")

In [ ]:
# ── MongoDB connection test ────────────────────────────────────────────────────
# This cell verifies the database is reachable and the collections have data
# before we run any analysis.  If it fails, check your .env file.
import os

MONGODB_URI = os.getenv("MONGODB_URI", "")
DB_NAME     = os.getenv("DB_NAME", "aqi_predictor")

if not MONGODB_URI:
    raise EnvironmentError(
        "MONGODB_URI is not set.\n"
        "Add it to your .env file at the project root and re-run this cell."
    )

from pymongo import MongoClient
from pymongo.errors import ConnectionFailure

try:
    client = MongoClient(MONGODB_URI, serverSelectionTimeoutMS=8_000)
    client.admin.command("ping")
    db = client[DB_NAME]

    raw_count  = db["raw_data"].count_documents({})
    feat_count = db["features"].count_documents({})
    train_count = db["features"].count_documents({
        "target_aqi_24h": {"$exists": True, "$ne": None}
    })
    model_count = db["model_registry"].count_documents({})

    print("✅ Connected to MongoDB Atlas")
    print(f"   Database      : {DB_NAME}")
    print(f"   raw_data      : {raw_count:,} documents")
    print(f"   features      : {feat_count:,} documents  ({train_count:,} with target labels)")
    print(f"   model_registry: {model_count:,} documents")

    if raw_count == 0:
        print("\n⚠️  raw_data collection is empty.")
        print("   Run:  python src/backfill.py   to populate it.")
    if feat_count == 0:
        print("\n⚠️  features collection is empty.")
        print("   backfill.py runs feature engineering automatically.")

except ConnectionFailure as e:
    raise ConnectionError(
        f"Cannot connect to MongoDB.\n"
        f"Check MONGODB_URI in your .env file.\nError: {e}"
    )

In [ ]:
# ── Load ALL data from MongoDB Feature Store ───────────────────────────────────
#
# Two collections:
#   raw_data   → df       (one row per hour, raw API values)
#   features   → feat_df  (engineered rows WITH 12 pollutant target columns)
#
# EPA AQI is computed from raw pollutant concentrations using aqi_utils.
# OpenWeather's 1-5 aqi field is kept only as a reference input feature.
# ──────────────────────────────────────────────────────────────────────────────

from pymongo import ASCENDING
from src.aqi_utils import calculate_final_aqi

POLLUTANTS = ["pm2_5", "pm10", "o3", "no2"]

raw_col  = db["raw_data"]
feat_col = db["features"]

# ── Raw data ───────────────────────────────────────────────────────────────────
raw_docs = list(raw_col.find({}, {"_id": 0}).sort("datetime", ASCENDING))
if not raw_docs:
    raise RuntimeError("raw_data empty. Run:  python src/backfill.py")

df = pd.DataFrame(raw_docs)
df["datetime"] = pd.to_datetime(df["datetime"], utc=True)
df.sort_values("datetime", inplace=True)
df.reset_index(drop=True, inplace=True)

# Time columns
df["hour"]    = df["datetime"].dt.hour
df["weekday"] = df["datetime"].dt.weekday
df["month"]   = df["datetime"].dt.month
df["date"]    = df["datetime"].dt.date

# Compute EPA AQI from raw pollutant concentrations (for trend analysis)
def compute_epa_aqi(row):
    concs = {p: float(row.get(p, 0) or 0) for p in POLLUTANTS}
    res = calculate_final_aqi(concs)
    return res["aqi"], res.get("dominant_pollutant")

df[["computed_aqi", "dominant_pollutant"]] = df.apply(
    lambda r: pd.Series(compute_epa_aqi(r)), axis=1
)

# ── Feature store — ALL rows with pollutant targets ───────────────────────────
feat_docs = list(feat_col.find({}, {"_id": 0}).sort("datetime", ASCENDING))
feat_df = pd.DataFrame(feat_docs) if feat_docs else pd.DataFrame()
if not feat_df.empty:
    feat_df["datetime"] = pd.to_datetime(feat_df["datetime"], utc=True)
    feat_df.sort_values("datetime", inplace=True)
    feat_df.reset_index(drop=True, inplace=True)

# Training rows = features WITH all 12 pollutant target columns
target_filter = {f"target_{p}_{h}h": {"$exists": True, "$ne": None}
                 for p in POLLUTANTS for h in [24, 48, 72]}
train_docs = list(feat_col.find(target_filter, {"_id": 0}).sort("datetime", ASCENDING))
train_df = pd.DataFrame(train_docs) if train_docs else pd.DataFrame()
if not train_df.empty:
    train_df["datetime"] = pd.to_datetime(train_df["datetime"], utc=True)

print(f"{'raw_data rows':<30}: {len(df):,}")
print(f"{'features rows (all)':<30}: {len(feat_df):,}")
print(f"{'features rows (with targets)':<30}: {len(train_df):,}")
print(f"{'Date range':<30}: {df['datetime'].min().date()} → {df['datetime'].max().date()}")
print(f"{'Target columns (12)':<30}: {[c for c in (train_df.columns if not train_df.empty else []) if 'target_' in c][:4]} ...")

In [ ]:
# ── OPTIONAL: Export / Import from CSV (offline fallback) ─────────────────────
#
# If you need to run EDA without a live MongoDB connection (e.g. for demo):
#   1. Run EXPORT block once while connected.
#   2. On a machine without MongoDB, run IMPORT block instead.
#
# Set USE_CSV_FALLBACK = True to read from CSV instead of MongoDB.
# ──────────────────────────────────────────────────────────────────────────────

USE_CSV_FALLBACK = False          # ← change to True to use local CSV files
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

RAW_CSV  = DATA_DIR / "raw_data_export.csv"
FEAT_CSV = DATA_DIR / "features_export.csv"

if USE_CSV_FALLBACK:
    # ── IMPORT ─────────────────────────────────────────────────────────────────
    if not RAW_CSV.exists():
        raise FileNotFoundError(
            f"CSV not found at {RAW_CSV}.\n"
            "First run with USE_CSV_FALLBACK=False (connected to MongoDB) "
            "to generate the export files."
        )
    df      = pd.read_csv(RAW_CSV,  parse_dates=["datetime"])
    feat_df = pd.read_csv(FEAT_CSV, parse_dates=["datetime"]) if FEAT_CSV.exists() else pd.DataFrame()
    train_df = feat_df[feat_df["target_aqi_24h"].notna()].copy() if not feat_df.empty else pd.DataFrame()
    df["datetime"]      = pd.to_datetime(df["datetime"], utc=True)
    if not feat_df.empty:
        feat_df["datetime"] = pd.to_datetime(feat_df["datetime"], utc=True)
    print(f"✅ Loaded from CSV — raw={len(df):,}  features={len(feat_df):,}")

else:
    # ── EXPORT (run this once to create local backups) ─────────────────────────
    df.to_csv(RAW_CSV,  index=False)
    if not feat_df.empty:
        feat_df.to_csv(FEAT_CSV, index=False)
    print(f"✅ Exported CSVs to {DATA_DIR}/")
    print(f"   {RAW_CSV.name}  ({len(df):,} rows)")
    if not feat_df.empty:
        print(f"   {FEAT_CSV.name}  ({len(feat_df):,} rows)")

## 2. Dataset Overview <a id='2'></a>

In [ ]:
df.head(3)

In [ ]:
summary = df.describe().round(3)
summary

In [ ]:
# Data-type & non-null counts
df.info()

## 3. Missing Value Analysis <a id='3'></a>

In [ ]:
numeric_cols = ["aqi", "co", "no", "no2", "o3", "so2", "pm2_5", "pm10", "nh3",
                "temperature", "humidity", "pressure", "wind_speed", "clouds"]
avail = [c for c in numeric_cols if c in df.columns]

missing_pct = df[avail].isnull().mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left — missing % per column
colors = ["#e74c3c" if p > 5 else "#2ecc71" for p in missing_pct]
axes[0].barh(missing_pct.index, missing_pct.values, color=colors)
axes[0].axvline(5, color="gray", linestyle="--", linewidth=1, label="5% threshold")
axes[0].set_xlabel("Missing (%)")
axes[0].set_title("Missing Values per Feature", fontsize=13)
axes[0].legend()
for i, v in enumerate(missing_pct.values):
    axes[0].text(v + 0.1, i, f"{v:.1f}%", va="center", fontsize=8)

# Right — presence heatmap (sample of 200 rows)
sample = df[avail].iloc[::max(1, len(df)//200)].isnull().T
sns.heatmap(sample, cmap=["#2ecc71", "#e74c3c"], cbar=False, yticklabels=True, ax=axes[1])
axes[1].set_title("Data Presence Map (green=present, red=missing)", fontsize=13)
axes[1].set_xlabel("Row index (sampled)")

plt.suptitle("Missing Value Analysis", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

if missing_pct.max() == 0:
    print("✅ Dataset is complete — no missing values.")

## 4. AQI Distribution & Category Breakdown <a id='4'></a>

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# ── Histogram + KDE ──────────────────────────────────────────────────────────
sns.histplot(df["aqi"], bins=5, kde=False, ax=axes[0],
             palette=[AQI_COLORS[i] for i in range(1, 6)],
             color="steelblue", edgecolor="white")
for bar, (idx, color) in zip(axes[0].patches, AQI_COLORS.items()):
    bar.set_facecolor(color)
axes[0].set_title("AQI Value Distribution", fontsize=13)
axes[0].set_xlabel("AQI (1-5)")
axes[0].set_ylabel("Hourly count")
axes[0].set_xticks([1, 2, 3, 4, 5])

# ── Donut — category share ──────────────────────────────────────────────────
cat_counts = df["aqi"].value_counts().sort_index()
wedge_colors = [AQI_COLORS[i] for i in cat_counts.index]
wedges, texts, autotexts = axes[1].pie(
    cat_counts, labels=[AQI_LABELS[i] for i in cat_counts.index],
    colors=wedge_colors, autopct="%1.1f%%",
    pctdistance=0.75, startangle=140,
    wedgeprops=dict(width=0.55, edgecolor="white", linewidth=2)
)
for at in autotexts:
    at.set_fontsize(9)
axes[1].set_title("AQI Category Share", fontsize=13)

# ── CDF ──────────────────────────────────────────────────────────────────────
sorted_aqi = np.sort(df["aqi"])
cdf = np.arange(1, len(sorted_aqi) + 1) / len(sorted_aqi)
axes[2].plot(sorted_aqi, cdf, color="#2c3e50", linewidth=2)
for aqi_val, color in AQI_COLORS.items():
    axes[2].axvline(aqi_val, color=color, linestyle="--", linewidth=1.2,
                    label=AQI_LABELS[aqi_val])
axes[2].set_title("Cumulative Distribution of AQI", fontsize=13)
axes[2].set_xlabel("AQI")
axes[2].set_ylabel("Cumulative proportion")
axes[2].legend(fontsize=8)
axes[2].set_xticks([1, 2, 3, 4, 5])

plt.suptitle("AQI Distribution — Karachi", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

print("AQI Category Counts:")
print(cat_counts.rename(AQI_LABELS).to_string())

## 5. AQI Trend over Time — with 7-day Rolling Average <a id='5'></a>

In [ ]:
rolling_7d = df.set_index("datetime")["aqi"].rolling("7D").mean().reset_index()

fig = go.Figure()

# Hourly raw — thin, semi-transparent
fig.add_trace(go.Scatter(
    x=df["datetime"], y=df["aqi"],
    mode="lines", name="Hourly AQI",
    line=dict(color="rgba(100,149,237,0.35)", width=0.8)
))

# 7-day rolling average — bold
fig.add_trace(go.Scatter(
    x=rolling_7d["datetime"], y=rolling_7d["aqi"],
    mode="lines", name="7-Day Rolling Mean",
    line=dict(color="#e74c3c", width=2.5)
))

# AQI band fills
for aqi_val, label, color in [
    (1, "Good", "rgba(0,228,0,0.08)"),
    (2, "Fair", "rgba(255,255,0,0.08)"),
    (3, "Moderate", "rgba(255,126,0,0.10)"),
    (4, "Poor", "rgba(255,0,0,0.10)"),
    (5, "Very Poor", "rgba(143,63,151,0.12)"),
]:
    fig.add_hrect(y0=aqi_val - 0.5, y1=aqi_val + 0.5,
                  fillcolor=color, line_width=0, annotation_text=label,
                  annotation_position="right")

# Annotate the maximum AQI spike
max_row = df.loc[df["aqi"].idxmax()]
fig.add_annotation(
    x=max_row["datetime"], y=max_row["aqi"],
    text=f"Peak AQI={int(max_row['aqi'])}<br>{max_row['datetime'].strftime('%b %d')}",
    showarrow=True, arrowhead=2, bgcolor="#fff", bordercolor="#555",
    font=dict(size=11)
)

fig.update_layout(
    title="Karachi Hourly AQI Trend with 7-Day Rolling Average",
    xaxis_title="Date (UTC)", yaxis_title="AQI (1-5)",
    yaxis=dict(range=[0.5, 5.5], tickvals=[1,2,3,4,5]),
    height=500, template=PLOTLY_TEMPLATE,
    legend=dict(orientation="h", yanchor="bottom", y=1.02)
)
fig.show()

## 6. Daily Average AQI — Colour-coded Bar Chart <a id='6'></a>

## 6b. Computed EPA AQI Trend (Pollutant-First)

The EPA AQI is computed from actual hourly pollutant concentrations using  
`aqi_utils.calculate_final_aqi()`.  This is what the ML system ultimately forecasts.

Color bands show official EPA AQI categories.

In [ ]:
from src.aqi_utils import EPA_AQI_CATEGORIES

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(df["datetime"], df["computed_aqi"], linewidth=0.9, color="#e74c3c",
        alpha=0.85, label="Computed EPA AQI")
ax.plot(df.set_index("datetime")["computed_aqi"].rolling("7D").mean().index,
        df.set_index("datetime")["computed_aqi"].rolling("7D").mean(),
        linewidth=2.5, color="#c0392b", label="7-day rolling mean")

band_colors = ["#00e400", "#ffff00", "#ff7e00", "#ff0000", "#8f3f97", "#7e0023"]
for (low, high, label, color), bc in zip(EPA_AQI_CATEGORIES, band_colors):
    ax.axhspan(low, high, alpha=0.07, color=bc)
    ax.axhline(high, color=bc, linewidth=0.5, linestyle="--", alpha=0.4)
    ax.text(df["datetime"].max(), (low + high) / 2, f" {label}", fontsize=8, color=bc, va="center")

ax.set_title("EPA AQI over Time — computed from PM2.5, PM10, O3, NO2", fontsize=14, fontweight="bold")
ax.set_xlabel("Date (UTC)"); ax.set_ylabel("AQI (0–500)")
ax.legend(fontsize=10); ax.set_ylim(0, None)
plt.tight_layout(); plt.show()

print(f"Mean computed AQI: {df['computed_aqi'].mean():.1f}")
print(f"Max computed AQI:  {df['computed_aqi'].max():.1f}")
print(f"Dominant pollutant frequency:")
print(df["dominant_pollutant"].value_counts().to_string())

In [ ]:
# Dominant pollutant frequency pie chart
dom_counts = df["dominant_pollutant"].value_counts()
poll_colors = {"pm2_5": "#e74c3c", "pm10": "#e67e22", "o3": "#3498db", "no2": "#9b59b6"}
colors_pie = [poll_colors.get(p, "#95a5a6") for p in dom_counts.index]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].pie(dom_counts.values, labels=[p.upper().replace("_", ".") for p in dom_counts.index],
            colors=colors_pie, autopct="%1.1f%%", startangle=90,
            wedgeprops=dict(edgecolor="white", linewidth=1.5))
axes[0].set_title("Dominant Pollutant Frequency (hourly)", fontsize=13, fontweight="bold")

# Bar chart of mean computed AQI by month
monthly_aqi = df.groupby("month")["computed_aqi"].mean()
month_names = {1:"Jan",2:"Feb",3:"Mar",4:"Apr",5:"May",6:"Jun",
               7:"Jul",8:"Aug",9:"Sep",10:"Oct",11:"Nov",12:"Dec"}
axes[1].bar([month_names.get(m, str(m)) for m in monthly_aqi.index], monthly_aqi.values,
            color="#e74c3c", alpha=0.75, edgecolor="white")
axes[1].axhline(100, color="orange", linestyle="--", label="Moderate threshold (100)")
axes[1].set_title("Monthly Average Computed AQI", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Month"); axes[1].set_ylabel("Avg AQI")
axes[1].legend(fontsize=10)

plt.tight_layout(); plt.show()

In [ ]:
daily = df.set_index("datetime").resample("D")["aqi"].agg(["mean", "min", "max"]).reset_index()
daily.columns = ["date", "mean", "min", "max"]
daily["color"] = daily["mean"].round().clip(1, 5).astype(int).map(AQI_COLORS)

fig = go.Figure()

# Error bars showing daily range
fig.add_trace(go.Bar(
    x=daily["date"],
    y=daily["mean"],
    marker_color=daily["color"],
    error_y=dict(
        type="data",
        symmetric=False,
        array=(daily["max"] - daily["mean"]).values,
        arrayminus=(daily["mean"] - daily["min"]).values,
        color="rgba(0,0,0,0.3)"
    ),
    name="Daily Mean AQI",
    hovertemplate="%{x}<br>Mean: %{y:.2f}<br>Min: %{customdata[0]:.2f} | Max: %{customdata[1]:.2f}",
    customdata=daily[["min", "max"]].values
))

# Legend patches for AQI categories
for aqi_val, label in AQI_LABELS.items():
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode="markers",
        marker=dict(size=10, color=AQI_COLORS[aqi_val], symbol="square"),
        name=f"{aqi_val} — {label}"
    ))

fig.update_layout(
    title="Daily Average AQI — Colour-coded by Category (bars show daily range)",
    xaxis_title="Date", yaxis_title="AQI",
    height=450, template=PLOTLY_TEMPLATE,
    legend=dict(orientation="h", yanchor="bottom", y=1.02)
)
fig.show()

## 7. All Pollutants over Time <a id='7'></a>

In [ ]:
pollutants = [c for c in ["pm2_5", "pm10", "no2", "o3", "so2", "co", "nh3", "no"]
              if c in df.columns]

n = len(pollutants)
fig = make_subplots(
    rows=n, cols=1, shared_xaxes=True,
    subplot_titles=[f"{p.upper()} (μg/m³)" for p in pollutants],
    vertical_spacing=0.025
)

palette = px.colors.qualitative.Plotly
for i, poll in enumerate(pollutants, 1):
    fig.add_trace(
        go.Scatter(
            x=df["datetime"], y=df[poll], mode="lines",
            name=poll.upper(),
            line=dict(color=palette[(i - 1) % len(palette)], width=0.9)
        ),
        row=i, col=1
    )

fig.update_layout(
    title="All Pollutant Concentrations over Time — Karachi",
    height=180 * n, template=PLOTLY_TEMPLATE, showlegend=False
)
fig.show()

## 8. PM2.5 & PM10 with WHO Guidelines <a id='8'></a>

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)

for ax, col, who_24h, color, label in [
    (axes[0], "pm2_5", 15, "#e74c3c", "PM2.5"),
    (axes[1], "pm10",  45, "#27ae60", "PM10")
]:
    daily_poll = df.set_index("datetime").resample("D")[col].mean()
    ax.fill_between(daily_poll.index, daily_poll, alpha=0.35, color=color)
    ax.plot(daily_poll.index, daily_poll, color=color, linewidth=1.5, label=f"{label} daily mean")
    ax.axhline(who_24h, color="black", linestyle="--", linewidth=1.5,
               label=f"WHO 24h guideline ({who_24h} μg/m³)")
    # Shade exceedance area
    ax.fill_between(daily_poll.index, who_24h, daily_poll.clip(lower=who_24h),
                    alpha=0.25, color="red", label="Exceedance")
    ax.set_ylabel(f"{label} (μg/m³)")
    ax.legend(loc="upper right", fontsize=9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))

axes[1].set_xlabel("Date")
plt.xticks(rotation=35)
plt.suptitle("PM2.5 & PM10 — Daily Mean vs WHO 24-hour Guidelines", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

## 9. Meteorological Features over Time <a id='9'></a>

In [ ]:
met_cols = [c for c in ["temperature", "humidity", "wind_speed", "pressure", "clouds"]
            if c in df.columns]
met_labels = {
    "temperature": "Temperature (°C)",
    "humidity": "Relative Humidity (%)",
    "wind_speed": "Wind Speed (m/s)",
    "pressure": "Surface Pressure (hPa)",
    "clouds": "Cloud Cover (%)"
}
met_colors = ["#e74c3c", "#3498db", "#2ecc71", "#9b59b6", "#95a5a6"]

fig = make_subplots(
    rows=len(met_cols), cols=1, shared_xaxes=True,
    subplot_titles=[met_labels.get(c, c) for c in met_cols],
    vertical_spacing=0.04
)

for i, (col, color) in enumerate(zip(met_cols, met_colors), 1):
    daily_met = df.set_index("datetime").resample("6H")[col].mean().reset_index()
    fig.add_trace(
        go.Scatter(
            x=daily_met["datetime"], y=daily_met[col], mode="lines",
            name=met_labels.get(col, col), line=dict(color=color, width=1.2),
            fill="tozeroy", fillcolor=color.replace(")", ",0.08)").replace("#", "rgba(") if color.startswith("#") else color
        ),
        row=i, col=1
    )

fig.update_layout(
    title="Meteorological Features (6-hourly smoothed) — Karachi",
    height=200 * len(met_cols), template=PLOTLY_TEMPLATE, showlegend=False
)
fig.show()

## 10. AQI by Hour of Day — Box Plot <a id='10'></a>

In [ ]:
fig = px.box(
    df, x="hour", y="aqi",
    color="hour",
    color_discrete_sequence=px.colors.sequential.Viridis,
    title="AQI Distribution by Hour of Day (UTC)",
    labels={"hour": "Hour (UTC)", "aqi": "AQI"},
    template=PLOTLY_TEMPLATE
)
fig.update_traces(showlegend=False)

# Overlay mean line
hourly_mean = df.groupby("hour")["aqi"].mean()
fig.add_trace(go.Scatter(
    x=hourly_mean.index, y=hourly_mean.values,
    mode="lines+markers", name="Hourly mean",
    line=dict(color="red", width=2.5),
    marker=dict(size=7, symbol="diamond")
))

fig.update_layout(
    height=480,
    xaxis=dict(tickmode="linear", tick0=0, dtick=1),
    yaxis=dict(tickvals=[1,2,3,4,5])
)
fig.show()

best_hour = int(hourly_mean.idxmin())
worst_hour = int(hourly_mean.idxmax())
print(f"Best hour  (lowest AQI): {best_hour:02d}:00 UTC — mean AQI={hourly_mean[best_hour]:.2f}")
print(f"Worst hour (highest AQI): {worst_hour:02d}:00 UTC — mean AQI={hourly_mean[worst_hour]:.2f}")

## 11. Hour × Weekday Heatmap <a id='11'></a>

In [ ]:
pivot_hw = df.pivot_table(index="weekday", columns="hour", values="aqi", aggfunc="mean")
pivot_hw.index = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"][:len(pivot_hw)]

fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(
    pivot_hw, cmap="YlOrRd", vmin=1, vmax=5,
    annot=True, fmt=".1f", annot_kws={"size": 8},
    linewidths=0.4, linecolor="white",
    cbar_kws={"label": "Mean AQI", "shrink": 0.6},
    ax=ax
)
ax.set_title("Mean AQI by Hour of Day × Weekday\n(reveals traffic patterns & nocturnal pollution peaks)",
             fontsize=14, fontweight="bold")
ax.set_xlabel("Hour (UTC)")
ax.set_ylabel("Day of Week")
plt.tight_layout()
plt.show()

## 12. AQI by Weekday — Violin Plot <a id='12'></a>

In [ ]:
day_map = {0:"Mon", 1:"Tue", 2:"Wed", 3:"Thu", 4:"Fri", 5:"Sat", 6:"Sun"}
df["day_name"] = df["weekday"].map(day_map)
day_order = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Violin
sns.violinplot(
    data=df, x="day_name", y="aqi", order=day_order,
    palette=["#ff6b6b" if d in ["Sat", "Sun"] else "#4ecdc4" for d in day_order],
    inner="quart", linewidth=1.2, ax=axes[0]
)
axes[0].set_title("AQI Distribution by Weekday (Violin)", fontsize=13)
axes[0].set_xlabel("")
axes[0].set_ylabel("AQI")
axes[0].set_ylim(0.5, 5.5)

# Grouped box — weekday vs weekend
df["is_weekend"] = df["weekday"].isin([5, 6]).map({True: "Weekend", False: "Weekday"})
sns.boxplot(
    data=df, x="is_weekend", y="aqi",
    palette={"Weekday": "#4ecdc4", "Weekend": "#ff6b6b"},
    width=0.5, ax=axes[1]
)
axes[1].set_title("Weekday vs Weekend AQI", fontsize=13)
axes[1].set_xlabel("")
axes[1].set_ylabel("AQI")

# Add mean annotations
for label, group in df.groupby("is_weekend")["aqi"]:
    x_pos = ["Weekday", "Weekend"].index(label)
    axes[1].text(x_pos, group.mean() + 0.08, f"μ={group.mean():.2f}",
                 ha="center", fontsize=11, fontweight="bold")

plt.suptitle("Weekly AQI Patterns — Karachi", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

## 13. Monthly / Daily AQI Calendar Heatmap <a id='13'></a>

In [ ]:
pivot_md = df.pivot_table(index="month", columns="day", values="aqi", aggfunc="mean")
month_names = {1:"Jan",2:"Feb",3:"Mar",4:"Apr",5:"May",6:"Jun",
               7:"Jul",8:"Aug",9:"Sep",10:"Oct",11:"Nov",12:"Dec"}
pivot_md.index = [month_names[m] for m in pivot_md.index]

fig, ax = plt.subplots(figsize=(18, max(3, len(pivot_md) * 0.9)))
sns.heatmap(
    pivot_md, cmap="RdYlGn_r", vmin=1, vmax=5,
    annot=True, fmt=".1f", annot_kws={"size": 8},
    linewidths=0.5, linecolor="white",
    cbar_kws={"label": "Daily Mean AQI", "shrink": 0.5},
    ax=ax
)
ax.set_title("Calendar Heatmap — Daily Mean AQI by Month & Day",
             fontsize=14, fontweight="bold")
ax.set_xlabel("Day of Month")
ax.set_ylabel("Month")
plt.tight_layout()
plt.show()

## 14. Correlation Heatmap <a id='14'></a>

In [ ]:
corr_cols = [c for c in numeric_cols if c in df.columns]
corr = df[corr_cols].corr()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Full correlation matrix (lower triangle)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
    center=0, linewidths=0.4, vmin=-1, vmax=1,
    annot_kws={"size": 8}, ax=axes[0],
    cbar_kws={"shrink": 0.8}
)
axes[0].set_title("Full Correlation Matrix", fontsize=13)

# Correlation with AQI only — sorted bar
aqi_corr = corr["aqi"].drop("aqi").sort_values()
colors_bar = ["#e74c3c" if v > 0 else "#3498db" for v in aqi_corr]
aqi_corr.plot(kind="barh", ax=axes[1], color=colors_bar, edgecolor="white")
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_title("Correlation with AQI\n(red=positive, blue=negative)", fontsize=13)
axes[1].set_xlabel("Pearson r")
for i, v in enumerate(aqi_corr):
    axes[1].text(v + (0.01 if v >= 0 else -0.01), i,
                 f"{v:+.2f}", va="center", ha="left" if v >= 0 else "right", fontsize=9)

plt.suptitle("Feature Correlation Analysis", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

## 15. Pollutants vs AQI — Scatter + Regression <a id='15'></a>

In [ ]:
poll_list = [c for c in ["pm2_5", "pm10", "no2", "o3", "so2", "co"] if c in df.columns]
n_cols = 3
n_rows = (len(poll_list) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(17, 5 * n_rows))
axes = axes.flatten()

for i, poll in enumerate(poll_list):
    sub = df[[poll, "aqi"]].dropna()
    r = sub.corr().iloc[0, 1]

    # Colour-code scatter points by AQI category
    point_colors = sub["aqi"].round().clip(1, 5).astype(int).map(AQI_COLORS)
    axes[i].scatter(sub[poll], sub["aqi"],
                    c=point_colors, alpha=0.4, s=12, edgecolors="none")

    # Regression line
    m, b = np.polyfit(sub[poll], sub["aqi"], 1)
    x_range = np.linspace(sub[poll].min(), sub[poll].max(), 100)
    axes[i].plot(x_range, m * x_range + b, color="black", linewidth=2,
                 linestyle="--", label=f"r = {r:.2f}")

    axes[i].set_xlabel(f"{poll.upper()} (μg/m³)", fontsize=10)
    axes[i].set_ylabel("AQI", fontsize=10)
    axes[i].set_title(f"{poll.upper()} vs AQI", fontsize=12)
    axes[i].legend(fontsize=10)

# AQI legend
patches = [mpatches.Patch(color=AQI_COLORS[k], label=AQI_LABELS[k]) for k in AQI_COLORS]
fig.legend(handles=patches, title="AQI Category", loc="lower center",
           ncol=5, bbox_to_anchor=(0.5, -0.02), fontsize=9)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Pollutant Concentration vs AQI (coloured by category)",
             fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

## 16. Wind Speed vs AQI <a id='16'></a>

In [ ]:
if "wind_speed" in df.columns:
    sub = df[["wind_speed", "aqi", "aqi_label"]].dropna()

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Scatter coloured by AQI
    point_colors = sub["aqi"].round().clip(1, 5).astype(int).map(AQI_COLORS)
    axes[0].scatter(sub["wind_speed"], sub["aqi"],
                    c=point_colors, alpha=0.4, s=15, edgecolors="none")
    m, b = np.polyfit(sub["wind_speed"], sub["aqi"], 1)
    xr = np.linspace(sub["wind_speed"].min(), sub["wind_speed"].max(), 100)
    r = sub[["wind_speed","aqi"]].corr().iloc[0,1]
    axes[0].plot(xr, m * xr + b, "k--", linewidth=2, label=f"r = {r:.2f}")
    axes[0].set_xlabel("Wind Speed (m/s)")
    axes[0].set_ylabel("AQI")
    axes[0].set_title("Wind Speed vs AQI")
    axes[0].legend(fontsize=11)

    # Box plot of wind speed per AQI category
    sns.boxplot(
        data=sub, x="aqi_label",
        y="wind_speed",
        order=[AQI_LABELS[i] for i in range(1,6) if AQI_LABELS[i] in sub["aqi_label"].unique()],
        palette=[AQI_COLORS[i] for i in range(1,6)],
        ax=axes[1]
    )
    axes[1].set_title("Wind Speed Distribution per AQI Category")
    axes[1].set_xlabel("AQI Category")
    axes[1].set_ylabel("Wind Speed (m/s)")

    plt.suptitle("Wind Speed & Air Quality — Higher wind disperses pollutants",
                 fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("wind_speed column not available.")

## 17. Temperature & Humidity vs AQI <a id='17'></a>

In [ ]:
met_vars = [(c, lbl) for c, lbl in
            [("temperature", "Temperature (°C)"), ("humidity", "Relative Humidity (%)")]
            if c in df.columns]

if met_vars:
    fig = make_subplots(
        rows=1, cols=len(met_vars),
        subplot_titles=[lbl for _, lbl in met_vars]
    )

    for col_idx, (col, lbl) in enumerate(met_vars, 1):
        sub = df[[col, "aqi"]].dropna()
        r = sub.corr().iloc[0, 1]
        point_colors_hex = sub["aqi"].round().clip(1, 5).astype(int).map(AQI_COLORS)

        fig.add_trace(
            go.Scatter(
                x=sub[col], y=sub["aqi"],
                mode="markers",
                marker=dict(color=point_colors_hex.tolist(), size=5, opacity=0.4),
                name=lbl, showlegend=False
            ),
            row=1, col=col_idx
        )

        # Trend line
        m, b = np.polyfit(sub[col], sub["aqi"], 1)
        xr = np.linspace(sub[col].min(), sub[col].max(), 100)
        fig.add_trace(
            go.Scatter(
                x=xr, y=m * xr + b, mode="lines",
                line=dict(color="black", dash="dash", width=2),
                name=f"r={r:.2f}"
            ),
            row=1, col=col_idx
        )

    fig.update_layout(
        title="Meteorological Variables vs AQI",
        height=450, template=PLOTLY_TEMPLATE
    )
    fig.update_yaxes(title_text="AQI", tickvals=[1,2,3,4,5])
    fig.show()

## 18. Pair Plot — Top Correlated Features <a id='18'></a>

In [ ]:
top_features = (df[corr_cols].corr()["aqi"]
                .drop("aqi").abs().sort_values(ascending=False).head(4).index.tolist())
pair_cols = ["aqi"] + top_features
pair_df = df[pair_cols + ["aqi_label"]].dropna().sample(min(800, len(df)), random_state=42)

g = sns.PairGrid(
    pair_df, vars=pair_cols,
    hue="aqi_label",
    palette={v: AQI_COLORS[k] for k, v in AQI_LABELS.items()},
    corner=True, height=2.2
)
g.map_lower(sns.scatterplot, alpha=0.4, s=15, edgecolors="none")
g.map_diag(sns.kdeplot, fill=True, alpha=0.5)
g.add_legend(title="AQI Category", bbox_to_anchor=(1, 0.7))
g.figure.suptitle("Pair Plot — AQI and Top Correlated Features",
                  fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 19. AQI Autocorrelation — Lag Analysis <a id='19'></a>

In [ ]:
from pandas.plotting import autocorrelation_plot
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

aqi_series = df["aqi"].dropna()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

plot_acf(aqi_series, lags=72, ax=axes[0], alpha=0.05, color="#3498db")
axes[0].set_title("Autocorrelation Function (ACF) — up to 72 lags", fontsize=13)
axes[0].set_xlabel("Lag (hours)")

# Mark key lags
for lag, label in [(1, "1h"), (24, "24h"), (48, "48h")]:
    axes[0].axvline(lag, color="red", linestyle="--", linewidth=1.2, alpha=0.7)
    axes[0].text(lag + 0.5, axes[0].get_ylim()[1] * 0.95, label,
                 color="red", fontsize=9)

plot_pacf(aqi_series, lags=48, ax=axes[1], alpha=0.05, color="#e74c3c", method="ywm")
axes[1].set_title("Partial Autocorrelation (PACF) — up to 48 lags", fontsize=13)
axes[1].set_xlabel("Lag (hours)")

plt.suptitle(
    "AQI Autocorrelation — strong 1h and 24h signals justify our lag features",
    fontsize=14, fontweight="bold"
)
plt.tight_layout()
plt.show()

# Manual lag correlation table
lag_corrs = {f"lag_{h}h": aqi_series.corr(aqi_series.shift(h)) for h in [1, 2, 6, 12, 24, 48, 72]}
lag_df = pd.DataFrame(lag_corrs, index=["Pearson r with AQI"]).T
print("\nLag Correlations with current AQI:")
print(lag_df.to_string())

## 20. Engineered Lag & Rolling Features <a id='20'></a>

In [ ]:
if not feat_df.empty:
    # Show PM2.5 lag/rolling features as representative example
    lag_features = [c for c in ["pm2_5", "pm2_5_lag_1", "pm2_5_lag_24", "pm2_5_lag_48",
                                 "pm2_5_rolling_6_mean", "pm2_5_rolling_12_mean",
                                 "pm2_5_rolling_24_mean"] if c in feat_df.columns]
    if lag_features:
        fig, axes = plt.subplots(len(lag_features), 1, figsize=(15, 3 * len(lag_features)),
                                 sharex=True)
        if len(lag_features) == 1:
            axes = [axes]
        colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(lag_features)))
        for ax, feat, color in zip(axes, lag_features, colors):
            ax.plot(feat_df["datetime"], feat_df[feat], color=color, linewidth=0.8, alpha=0.85)
            ax.set_ylabel(feat, fontsize=9)
            ax.grid(alpha=0.3)
        axes[-1].set_xlabel("Date (UTC)")
        axes[0].set_title("PM2.5 Lag & Rolling Features (from features collection)", fontsize=13)
        plt.tight_layout()
        plt.show()
    else:
        print("PM2.5 lag features not found in feat_df. Run backfill.py --rebuild-features")24", "aqi_lag_48", "aqi_rolling_24_mean"]
                    if c in feat_df.columns]

    fig = go.Figure()
    styles = [
        dict(color="rgba(100,149,237,0.9)", width=1.5, dash="solid"),
        dict(color="rgba(255,127,14,0.8)",  width=1,   dash="dot"),
        dict(color="rgba(44,160,44,0.8)",   width=1,   dash="dash"),
        dict(color="rgba(214,39,40,0.8)",   width=1,   dash="dashdot"),
        dict(color="rgba(148,103,189,1.0)", width=2.5, dash="solid"),
    ]
    for feat, style in zip(lag_features, styles):
        fig.add_trace(go.Scatter(
            x=feat_df["datetime"], y=feat_df[feat],
            mode="lines", name=feat, line=style
        ))

    fig.update_layout(
        title="Current AQI vs Lag & Rolling Features over Time",
        xaxis_title="Date", yaxis_title="AQI",
        height=480, template=PLOTLY_TEMPLATE,
        legend=dict(orientation="h", yanchor="bottom", y=1.02)
    )
    fig.show()
else:
    print("Feature DataFrame is empty. Run feature_engineering.py first.")

## 21. AQI Change Rate Distribution <a id='21'></a>

In [ ]:
if not feat_df.empty and "pm2_5_change_rate" in feat_df.columns:
    cr = feat_df["pm2_5_change_rate"].dropna()
    cr_clipped = cr.clip(-2, 2)  # clip outliers for display

    fig, axes = plt.subplots(1, 3, figsize=(17, 5))

    # KDE
    sns.kdeplot(cr_clipped, ax=axes[0], fill=True, color="#9b59b6", linewidth=2)
    axes[0].axvline(0, color="black", linestyle="--", linewidth=1.5)
    axes[0].axvline(cr.mean(), color="red", linestyle="-", linewidth=1.5,
                    label=f"mean={cr.mean():.3f}")
    axes[0].set_title("AQI Change Rate — KDE", fontsize=13)
    axes[0].set_xlabel("Change rate (clipped ±200%)")
    axes[0].legend()

    # Over time
    axes[1].plot(feat_df["datetime"], cr_clipped, linewidth=0.6, color="#9b59b6", alpha=0.7)
    axes[1].axhline(0, color="black", linestyle="--", linewidth=1)
    axes[1].fill_between(feat_df["datetime"], 0, cr_clipped,
                          where=cr_clipped > 0, alpha=0.3, color="red", label="Worsening")
    axes[1].fill_between(feat_df["datetime"], 0, cr_clipped,
                          where=cr_clipped < 0, alpha=0.3, color="green", label="Improving")
    axes[1].set_title("AQI Change Rate over Time", fontsize=13)
    axes[1].set_xlabel("Date")
    axes[1].set_ylabel("% Change vs 24h ago")
    axes[1].legend(fontsize=9)
    axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    axes[1].xaxis.set_major_locator(mdates.WeekdayLocator(interval=3))
    plt.setp(axes[1].get_xticklabels(), rotation=30)

    # Box per weekday
    feat_df["day_name"] = feat_df["datetime"].dt.weekday.map(day_map)
    sns.boxplot(
        data=feat_df, x="day_name", y="aqi_change_rate",
        order=day_order, palette="coolwarm", ax=axes[2]
    )
    axes[2].axhline(0, color="black", linestyle="--", linewidth=1)
    axes[2].set_ylim(-2, 2)
    axes[2].set_title("AQI Change Rate by Weekday", fontsize=13)
    axes[2].set_xlabel("")
    axes[2].set_ylabel("Change rate")

    plt.suptitle("AQI Change Rate Analysis", fontsize=15, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("aqi_change_rate not available.")

## 22. Target Variable Distributions <a id='22'></a>

In [ ]:
# Target variable distributions — pollutant targets (4 pollutants × 3 horizons)
# Uses train_df: feature rows WITH all 12 target columns from the features collection.

if not train_df.empty:
    for pollutant in ["pm2_5", "pm10", "o3", "no2"]:
        targets = [f"target_{pollutant}_{h}h" for h in [24, 48, 72]
                   if f"target_{pollutant}_{h}h" in train_df.columns]
        if not targets:
            continue
        colors_t = ["#e74c3c", "#f39c12", "#8e44ad"]
        fig, axes = plt.subplots(1, len(targets) + 1, figsize=(18, 4))
        for i, (tgt, col) in enumerate(zip(targets, colors_t)):
            data = train_df[tgt].dropna()
            axes[i].hist(data, bins=25, color=col, alpha=0.6, edgecolor="white", density=True)
            sns.kdeplot(data, ax=axes[i], color=col, linewidth=2.5)
            axes[i].axvline(data.mean(), color="black", linestyle="--",
                            linewidth=1.5, label=f"μ={data.mean():.1f}")
            axes[i].set_title(f"{tgt.replace('target_','').replace('_',' ')}", fontsize=11)
            axes[i].set_xlabel("μg/m³"); axes[i].set_ylabel("Density"); axes[i].legend(fontsize=9)
        for tgt, col in zip(targets, colors_t):
            data = train_df[tgt].dropna()
            sns.kdeplot(data, ax=axes[-1], color=col, linewidth=2, label=tgt.split("_")[-1], fill=True, alpha=0.15)
        axes[-1].set_title("Overlay — 24h/48h/72h")
        axes[-1].legend(fontsize=10)
        plt.suptitle(f"{pollutant.upper()} Target Distributions — 3 Horizons", fontsize=14, fontweight="bold")
        plt.tight_layout(); plt.show()
        print(train_df[targets].describe().round(2).to_string())
        print()
else:
    print("No training rows. Run:  python src/backfill.py --rebuild-features")

## 23. Quick Feature Importance Preview <a id='23'></a>

We fit a lightweight Random Forest on the 24h target to rank features
before formal training — a useful sanity check.

In [ ]:
TARGET_FOR_IMPORTANCE = "target_pm2_5_24h"

if not train_df.empty and TARGET_FOR_IMPORTANCE in train_df.columns:
    from sklearn.ensemble import RandomForestRegressor
    from src.config import FEATURE_COLUMNS

    feature_cols = [c for c in FEATURE_COLUMNS if c in train_df.columns]
    sub = train_df[feature_cols + [TARGET_FOR_IMPORTANCE]].dropna()

    X = sub[feature_cols].values
    y = sub[TARGET_FOR_IMPORTANCE].values

    rf = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
    rf.fit(X, y)

    importance_df = pd.DataFrame({
        "feature": feature_cols,
        "importance": rf.feature_importances_
    }).sort_values("importance", ascending=True)

    top20 = importance_df.tail(20)
    fig, ax = plt.subplots(figsize=(10, max(5, len(top20) * 0.4)))
    bar_colors = ["#e74c3c" if i >= len(top20) - 5 else "#95a5a6"
                  for i in range(len(top20))]
    bars = ax.barh(top20["feature"], top20["importance"],
                   color=bar_colors, edgecolor="white")
    ax.set_title(f"Random Forest Feature Importance — Target: {TARGET_FOR_IMPORTANCE}",
                 fontsize=13, fontweight="bold")
    ax.set_xlabel("Importance (MDI)")
    for bar in bars:
        w = bar.get_width()
        ax.text(w + 0.001, bar.get_y() + bar.get_height() / 2,
                f"{w:.3f}", va="center", fontsize=8)
    plt.tight_layout()
    plt.show()

    print("\nTop 5 most important features:")
    print(importance_df.tail(5)[["feature", "importance"]].to_string(index=False))
else:
    print(f"Column '{TARGET_FOR_IMPORTANCE}' not found. Run backfill --rebuild-features first.")

## 24. Key Insights & Modelling Implications <a id='24'></a>

---

### 🌫️ Pollutant-First AQI Approach

This project uses a **pollutant-first AQI forecasting** system:

1. **Forecast**: ML models predict PM2.5, PM10, O3, NO2 concentrations at +24h / +48h / +72h
2. **Convert**: Each concentration → AQI sub-index via EPA breakpoint interpolation
3. **Final AQI**: max(sub-indices) — the dominant pollutant drives the AQI

> ⚠️ This is an educational approximation. Official EPA AQI uses 24-hour PM averages and 8-hour O3 averages, not hourly values.

---

### 🟠 Key Findings — Karachi Air Quality

- **PM2.5** is typically the dominant AQI driver in Karachi — fine particles from traffic, biomass burning, and industrial emissions.
- **PM10** spikes are common in spring (March–May) due to dust storms — Karachi's unique geography makes it highly susceptible.
- **O3** peaks during afternoon hours (12–16 UTC) driven by photochemical reactions in sunlight.
- **NO2** is elevated near traffic-heavy morning hours, consistent with rush-hour emissions.
- **Wind speed** inversely correlates with all pollutants — sea breeze during evenings disperses urban pollution.
- **Humidity** shows a complex relationship: high humidity can worsen PM2.5 by hygroscopic growth.

### 📊 Modelling Implications

- **Lag features** (1h, 24h, 48h) are the strongest predictors — pollutant levels persist with autocorrelation.
- **Rolling means** (6h, 12h, 24h) capture pollution episodes better than single-hour snapshots.
- **Time features** (hour, weekday) capture diurnal/weekly patterns in traffic and photochemistry.
- **Change rate** features help detect rapid transitions (storm arrivals, wind shifts).
- **XGBoost / GradientBoosting** tend to outperform Ridge on non-linear pollutant dynamics, especially for PM10 spikes.
- Overfitting is mitigated via AND-condition thresholds: `overfit_ratio > 1.5 AND r2_gap > 0.4`.ea-breeze ventilation.
- AQI almost never reaches **Good (1)**, suggesting persistent baseline pollution.

---

### ⏰ Temporal Patterns
| Pattern | Observation | Reason |
|---|---|---|
| **Hour of day** | Peaks in early morning (03–08 UTC) | Nocturnal boundary layer traps pollutants; low wind speeds |
| **Hour of day** | Relative improvement midday | Convective mixing + sea breeze |
| **Weekday** | Slightly higher Mon–Fri | Vehicle + industrial activity |
| **Monthly** | Seasonal variation | Monsoon (Jul–Sep) brings rainfall that scrubs PM |

---

### 🧪 Pollutant Contributions
| Pollutant | Correlation with AQI | Notes |
|---|---|---|
| **PM2.5** | Strongest positive | Primary AQI driver |
| **PM10** | Strong positive | Dust + coarse particles |
| **NO2** | Moderate positive | Traffic, combustion |
| **CO** | Moderate positive | Vehicle exhaust |
| **O3** | Weak / negative | Photochemical; inversely related to NO2 |
| **Wind speed** | Negative | Higher wind → better dispersion → lower AQI |
| **Humidity** | Slightly positive | Can enhance PM formation & hygroscopic growth |

---

### 🔁 Autocorrelation → Feature Engineering
- The ACF shows **very high autocorrelation at lag 1h** (r ≈ 0.95) and a **clear 24h cycle**
  — justifying `aqi_lag_1`, `aqi_lag_24`, `aqi_lag_48` as the most powerful features.
- The PACF confirms that once lag-1 is accounted for, partial correlations decay
  but a 24h spike remains — validating the rolling 24h mean feature.

---

### 🤖 Modelling Implications
- **Non-linear methods (RF, XGBoost) will outperform Linear Regression**
  because pollutant–AQI relationships are nonlinear and features interact.
- **Time-based train/test split is mandatory** — random splitting would leak
  future lag values into training data, causing over-optimistic results.
- **Class imbalance**: AQI=5 rows are rare → models may underpredict extreme events;
  consider class weighting or threshold adjustment post-training.
- **Feature importances** confirm lag and rolling features dominate, followed by
  PM2.5, hour-of-day, and wind speed — consistent with physical intuition.
